# Lesson 4c — Add hand-rolled attention

Phase 2 of the Karpathy track: build PRAGMA incrementally. Same data and task throughout L4a-L4d, with one piece added each lesson.

Previous lesson result: **L4b emb+linear: 64.2% acc, 0.558 CE (slight improvement over counts)**

Runnable version of `04c_*.py`.


## Same data and helpers as L4b

In [ ]:
import math, random, torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0); random.seed(0)

KEYS   = ["pet", "action", "place"]
VALUES = ["dog", "cat", "fish", "eat", "sleep", "play", "garden", "couch", "bowl"]
PAD, MASK = "<pad>", "<mask>"
vocab  = [PAD, MASK] + KEYS + VALUES
tok2id = {t: i for i, t in enumerate(vocab)}
V = len(vocab)
RULES = {
    "dog":  {"action": ["eat", "play"],   "place": ["garden", "bowl"]},
    "cat":  {"action": ["sleep", "play"], "place": ["couch", "bowl"]},
    "fish": {"action": ["eat", "sleep"],  "place": ["bowl"]},
}

def random_event():
    pet = random.choice(list(RULES))
    act = random.choice(RULES[pet]["action"])
    plc = random.choice(RULES[pet]["place"])
    return {"pet": pet, "action": act, "place": plc}

def encode_event(ev):
    ids = []
    for k in KEYS:
        ids.append(tok2id[k]); ids.append(tok2id[ev[k]])
    return ids

def make_mlm_example(ev, mask_field):
    ids = encode_event(ev)
    pos = 2 * KEYS.index(mask_field) + 1
    tgt = ids[pos]; ids[pos] = tok2id[MASK]
    return ids, pos, tgt

def build_dataset(events):
    Xs, positions, ys = [], [], []
    for ev in events:
        for k in KEYS:
            ids, pos, tgt = make_mlm_example(ev, k)
            Xs.append(ids); positions.append(pos); ys.append(tgt)
    return (torch.tensor(Xs, dtype=torch.long),
            torch.tensor(positions, dtype=torch.long),
            torch.tensor(ys, dtype=torch.long))

train = [random_event() for _ in range(5000)]
test  = [random_event() for _ in range(1000)]
X_tr, pos_tr, y_tr = build_dataset(train)
X_te, pos_te, y_te = build_dataset(test)

## Model — single-head self-attention layer (from L3c) on top of embeddings

This time we use the OUTPUT at the MASK POSITION, not mean-pool.

In [ ]:
D = 16

class SelfAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.d = d
        self.W_q = nn.Linear(d, d, bias=False)
        self.W_k = nn.Linear(d, d, bias=False)
        self.W_v = nn.Linear(d, d, bias=False)
        self.W_o = nn.Linear(d, d, bias=False)

    def forward(self, x):
        Q, K, V_ = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d)
        attn = F.softmax(scores, dim=-1)
        return self.W_o(attn @ V_), attn


class EmbAttentionModel(nn.Module):
    def __init__(self, V, d):
        super().__init__()
        self.emb  = nn.Embedding(V, d)
        self.attn = SelfAttention(d)
        self.head = nn.Linear(d, V)

    def forward(self, ids, positions=None):
        h = self.emb(ids)
        h, attn = self.attn(h)
        if positions is not None:
            B = h.size(0)
            mask_h = h[torch.arange(B), positions]  # Output at the mask position only.
        else:
            mask_h = h.mean(dim=1)
        return self.head(mask_h), attn


model = EmbAttentionModel(V, D)
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

## Train

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

BATCH, N_STEPS = 128, 3000
for step in range(N_STEPS):
    idx = torch.randint(0, X_tr.size(0), (BATCH,))
    logits, _ = model(X_tr[idx], pos_tr[idx])
    loss = loss_fn(logits, y_tr[idx])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 600 == 0:
        print(f"  step {step:4d}   loss {loss.item():.3f}")

## Evaluate

In [ ]:
model.eval()
with torch.no_grad():
    logits, _ = model(X_te, pos_te)
    pred = logits.argmax(-1)
    acc = (pred == y_te).float().mean().item()
    ce  = loss_fn(logits, y_te).item()

print(f"  Test accuracy:       {acc * 100:.1f}%")
print(f"  Test cross-entropy:  {ce:.3f}")
print()
print(f"  L4a (counts):           63.4% acc,  0.670 CE")
print(f"  L4b (emb+linear):       64.2% acc,  0.558 CE")
print(f"  L4c (emb+ATTENTION):    {acc * 100:.1f}% acc,  {ce:.3f} CE")

## Surprise: attention alone is WORSE than mean-pool

Why? Naïve attention with no residual connection, no FFN, no LayerNorm is hard to train. The attention picks ONE highly-predictive token but can't combine multiple tokens without an FFN.

**This is the motivation for L4d** — adding residuals, FFN, and LayerNorm.

[**L4d**](lesson_04d_full_block.ipynb) — full Transformer block with all the pieces.